<a href="https://colab.research.google.com/github/yufeixiao/grazer/blob/main/fire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install earthaccess

In [ ]:
import earthaccess
import pandas as pd

# 1. Authenticate
auth = earthaccess.login(strategy="interactive")

# 2. Define your "Search Area" (Bounding Box for the region near 35°S, 150°E)
# Format: (lower_left_lon, lower_left_lat, upper_right_lon, upper_right_lat)
bbox = (149.5, -36.0, 151.5, -34.5)

# 3. Search for the 16-day NDVI (250m resolution)
# Comparing Jan 2019 (Before) and Jan 2020 (During/Burn Scar)
ndvi_results = earthaccess.search_data(
    short_name="MYD13Q1",
    cloud_hosted=True,
    bounding_box=bbox,
    temporal=("2019-01-01", "2020-02-01"),
    count=10
)

# 4. Search for the Daily Surface Reflectance (Bands 7, 2, 1)
reflectance_results = earthaccess.search_data(
    short_name="MYD09GA",
    cloud_hosted=True,
    bounding_box=bbox,
    temporal=("2020-01-10", "2020-01-25"), # Specific fire window
    count=5
)

# 5. List the download links for the first result
if reflectance_results:
    print(f"Found {len(reflectance_results)} reflectance files.")
    print("Download link for first file:", reflectance_results[0].data_links()[0])

Enter your Earthdata Login username: graphfei
Enter your Earthdata password: ··········
Found 5 reflectance files.
Download link for first file: https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/MYD09GA.061/MYD09GA.A2020010.h31v12.061.2020325014845/MYD09GA.A2020010.h31v12.061.2020325014845.hdf


In [ ]:
!pip install leafmap
import leafmap

In [ ]:
import leafmap
m = leafmap.Map(center=[-35.14, 150.59], zoom=9)
# Add the MODIS Aqua 7-2-1 False Color layer for Jan 20, 2020
m.add_wms_layer(
    url="https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi",
    layers="MODIS_Aqua_CorrectedReflectance_Bands721",
    time="2020-01-20",
    name="MODIS 7-2-1 False Color"
)
m

Map(center=[-35.14, 150.59], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

# landsat

In [ ]:
import earthaccess

# 1. Login
auth = earthaccess.login(strategy="interactive")

# 2. Define Area (Near Nowra/Jervis Bay, NSW)
# Format: (lower_left_lon, lower_left_lat, upper_right_lon, upper_right_lat)
bbox = (150.4, -35.2, 150.8, -34.9)

# 3. Search for Landsat 8/9 Level 2 (Surface Reflectance)
# Short Name: LC08_L2SP (Landsat 8) or LC09_L2SP (Landsat 9)
landsat_results = earthaccess.search_data(
    short_name="LC08_L2SP",
    cloud_hosted=True,
    bounding_box=bbox,
    temporal=("2020-01-10", "2020-01-31"), # Peak of the burn scar
    count=5
)

for granule in landsat_results:
    print(f"File: {granule.info()['title']}")
    # These contain the individual TIF bands (SR_B7, SR_B5, SR_B4)
    print(f"Links: {granule.data_links()[:3]} ...")

In [ ]:
import earthaccess

# Use the 'landsat' provider specifically if short_name fails
# We'll search for Landsat 8 Level 2 (Surface Reflectance)
results = earthaccess.search_data(
    short_name="LC08_L2SP_02", # Added _02 for Collection 2
    bounding_box=(150.0, -35.5, 151.0, -34.5),
    temporal=("2015-01-01", "2025-02-15"),
    count=10
)

print(f"Found {len(results)} granules.")
for g in results:
    print(g.info()['title'])

Found 0 granules.


In [ ]:
!pip install pystac-client planetary-computer stackstac-0.5.0 # Install if needed

import pystac_client
import planetary_computer
import stackstac
import xarray as xr

# 1. Connect to the STAC Catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# 2. Search for Landsat 8 (C2 L2 is the Atmosphere-Corrected version)
# We use the bbox for the Nowra / Jervis Bay area
bbox = [150.0, -35.5, 151.0, -34.5]
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime="2019-12-01/2020-02-01", # Peak fire window
    query={"eo:cloud_cover": {"lt": 20}} # Max 20% cloud
)

items = search.get_all_items()
print(f"Found {len(items)} scenes.")

# 3. Create a 'DataCube' of the Burn Scar Bands (7, 5, 4)
# assets=["swir22", "nir08", "red"] matches Landsat 8 Bands 7, 5, 4
stack = stackstac.stack(items, assets=["swir22", "nir08", "red"], bounds_latlon=bbox)

# 4. View the specific dates found
print("Available Dates:", stack.time.values)

ERROR: Could not find a version that satisfies the requirement stackstac-0.5.0 (from versions: none)
ERROR: No matching distribution found for stackstac-0.5.0


ModuleNotFoundError: No module named 'planetary_computer'

In [ ]:
import leafmap

m = leafmap.Map(center=[-35.14, 150.59], zoom=10)

# This uses the Microsoft Planetary Computer STAC (very fast for Landsat)
m.add_stac_layer(collection="landsat-c2-l2",
    assets=["SR_B7", "SR_B5", "SR_B4"], # The "7-5-4" Burn Scar combo
    name="Landsat Burn Scar",
    fit_bounds=True
)
m

ValueError: TiTiler endpoint error: [{'type': 'missing', 'loc': ['query', 'item'], 'msg': 'Field required', 'input': None}]

In [ ]:
landsat_results

[]

# fires metadata

In [ ]:
import earthaccess

# 1. Login (This will open a link to Earthdata Login)
auth = earthaccess.login(strategy="interactive")

# 2. Search using your specific CMR Concept ID
# Note: You can add temporal and bounding_box filters here
results = earthaccess.search_data(
    concept_id="C2389158955-ORNL_CLOUD",
    temporal=("2010-01-01", "2010-12-31"), # Example: Global Fire Atlas covers 2003-2016
    count=10 # Start with a small number to inspect
)

# 3. View the "locations" (Download URLs or S3 links)
for granule in results:
    print(granule.data_links())

Enter your Earthdata Login username: graphfei
Enter your Earthdata password: ··········
['https://data.ornldaac.earthdata.nasa.gov/protected/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_fire_line_monthly_2010.tif', 'https://data.ornldaac.earthdata.nasa.gov/public/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_fire_line_monthly_2010.tif.sha256']
['https://data.ornldaac.earthdata.nasa.gov/protected/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_speed_monthly_2010.tif', 'https://data.ornldaac.earthdata.nasa.gov/public/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_speed_monthly_2010.tif.sha256']
['https://data.ornldaac.earthdata.nasa.gov/protected/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_direction_yearly_2010.tif', 'https://data.ornldaac.earthdata.nasa.gov/public/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_direction_yearly_2010.tif.sha256']
['https://data.ornldaac.earthdata.nasa.gov/protected/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_spread_monthly_2010.tif',

In [ ]:
!pip install rioxarray

In [ ]:
import xarray as xr
import rioxarray # noqa: F401 - This import registers the 'rasterio' engine

# Get file-like objects for the granules
files = earthaccess.open(results)

# Open the first file to inspect metadata/coordinates, specifying the 'rasterio' engine
ds = xr.open_dataset(files[0], engine="rasterio")
print(ds)

QUEUEING TASKS | :   0%|          | 0/20 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/20 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/20 [00:00<?, ?it/s]

<xarray.Dataset> Size: 50MB
Dimensions:      (band: 12, x: 1440, y: 720)
Coordinates:
  * band         (band) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * x            (x) float64 12kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
  * y            (y) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float32 50MB ...


In [ ]:
import xarray as xr
import rioxarray # noqa: F401

# Identify the file for 'day of burn'
day_of_burn_file = None
for f_obj in files:
    if 'day_of_burn_yearly' in f_obj.url:
        day_of_burn_file = f_obj
        break

if day_of_burn_file:
    print(f"Opening: {day_of_burn_file.url}")
    day_of_burn_ds = xr.open_dataset(day_of_burn_file, engine="rasterio")
    print(day_of_burn_ds)
else:
    print("Could not find 'day_of_burn_yearly' file in the results.")

Opening: https://data.ornldaac.earthdata.nasa.gov/protected/cms/CMS_Global_Fire_Atlas/data/Global_fire_atlas_day_of_burn_yearly_2010.tif
<xarray.Dataset> Size: 20GB
Dimensions:      (band: 1, x: 81600, y: 31200)
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 653kB -2.001e+07 -2.001e+07 ... 1.779e+07 1.779e+07
  * y            (y) float64 250kB 7.783e+06 7.783e+06 ... -6.671e+06 -6.671e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float64 20GB ...


In [ ]:
import pandas as pd
import numpy as np

# Assuming day_of_burn_ds is already loaded from the previous step

# Extract the band_data (day of burn values)
day_of_burn_values = day_of_burn_ds['band_data'].squeeze() # Remove the 'band' dimension

# Identify pixels where a fire occurred (day of burn > 0)
# Using np.argwhere to get indices of non-zero elements efficiently
burned_indices = np.argwhere(day_of_burn_values.values > 0)

# If no fires found, print a message and exit
if burned_indices.size == 0:
    print("No fire events found in the dataset for 2010.")
else:
    print(f"Found {len(burned_indices)} fire events.")

    # Take a small sample to avoid processing huge arrays for demonstration
    sample_size = min(10, len(burned_indices))
    np.random.seed(42) # for reproducibility
    sample_indices = burned_indices[np.random.choice(len(burned_indices), sample_size, replace=False)]

    # Extract x, y coordinates and day of burn for the sample
    sample_y_indices, sample_x_indices = sample_indices[:, 0], sample_indices[:, 1]

    fire_x = day_of_burn_values.x.values[sample_x_indices]
    fire_y = day_of_burn_values.y.values[sample_y_indices]
    fire_day_of_year = day_of_burn_values.values[sample_y_indices, sample_x_indices]

    # Convert day of year to actual date in 2010
    year = 2010
    fire_dates = pd.to_datetime(year * 1000 + fire_day_of_year, format='%Y%j')

    # Create a DataFrame for better organization
    fire_events = pd.DataFrame({
        'x': fire_x,
        'y': fire_y,
        'day_of_year': fire_day_of_year,
        'date': fire_dates
    })

    print("\nSample of extracted fire events:")
    print(fire_events)

    # You can now use these 'fire_events' to define bounding boxes and temporal ranges
    # for searching spectral data. For example, to get the bounding box for the first fire:
    first_fire_x = fire_events.iloc[0]['x']
    first_fire_y = fire_events.iloc[0]['y']
    first_fire_date = fire_events.iloc[0]['date']
    print(first_fire_x,first_fire_y,first_fire_date)
